# Notebook 3 — Predict **P(correct)** per compression config

**LightGBM classification, 10-value output.** For a single prompt this model predicts the probability the answer is correct under each of the 10 compression configurations (Baseline full precision, KVQuant 2/3/4-bit, H2O 20/40/60%, RocketKV 8/16/32x) — a length-10 probability vector.

**Protocol.** Reshape → hold out **15%** test (stratified by dataset) → tune with **repeated stratified k-fold CV** (5 folds × 3 repeats) on the 85%, selecting the config with the lowest mean **log loss** (a proper scoring rule for probabilities) → refit on all 85% → evaluate the test set once with log loss, AUC, accuracy and Brier score.

## Setup

In [1]:
# Install deps (Colab-safe; no-op if already present). Add --break-system-packages
# locally if pip refuses to touch a managed environment.
!pip install -q lightgbm scikit-learn scipy pandas numpy joblib requests
import os, warnings
os.makedirs("artifacts", exist_ok=True)
warnings.filterwarnings("ignore", message="X does not have valid feature names")


## Engine — data loading, reshape, features

In [2]:
# ===========================================================================
# ENGINE — shared across the latency / memory / correctness notebooks.
# Only the CONFIG block below changes between the three notebooks.
# ===========================================================================
import io, os, re, time, itertools
from pathlib import Path
import numpy as np, pandas as pd, requests

# ------------------------------- CONFIG ------------------------------------
TARGET   = "correct"
ARTIFACT = "clf_correct_lightgbm.joblib"
# ---------------------------------------------------------------------------

RS = 42                    # global random seed — one split, reproducible everywhere
TEST_FRAC = 0.15           # 15% held out for the final test; 85% for train+val+CV

# The 10 compression configurations (Baseline = full precision, no compression).
# THIS is the output axis: every model emits a length-10 vector, one predicted
# value per configuration, for a single prompt.
CONFIGS = ["baseline",
           "kvquant_2bit", "kvquant_3bit", "kvquant_4bit",
           "h2o_20", "h2o_40", "h2o_60",
           "rocketkv_8x", "rocketkv_16x", "rocketkv_32x"]

# Map each config to the on-disk / on-repo filename stem.
FNAME = {"baseline": "kvquant_baseline_full_precision",
         "kvquant_2bit": "kvquant_2bit", "kvquant_3bit": "kvquant_3bit", "kvquant_4bit": "kvquant_4bit",
         "h2o_20": "h2o_budget_20pct", "h2o_40": "h2o_budget_40pct", "h2o_60": "h2o_budget_60pct",
         "rocketkv_8x": "rocketkv_ratio_8x", "rocketkv_16x": "rocketkv_ratio_16x", "rocketkv_32x": "rocketkv_ratio_32x"}

DATASETS = ["gsm8k", "arc_challenge", "hellaswag", "squad"]

# Load CSVs from Google Drive -- KVQuant_v3_Results is the authoritative
# source every implementation notebook actually writes to; the repo's
# 2048_sample_results2/ folder is just a checked-in COPY of it, kept as a
# fallback in case Drive isn't mounted (e.g. local Jupyter) or a file hasn't
# been re-copied there yet. Falls back further to a local ./Data checkout or
# raw GitHub as a last resort.
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass          # not running in Colab (e.g. local Jupyter) -- fall through

# Where the trained model artifact gets saved -- this MUST be the same Drive
# folder evaluate_adaptive_compression() loads from, or that notebook keeps
# reading a stale artifact no matter how many times this one reruns.
MODEL_DIR = Path("/content/drive/MyDrive/KV_Cache_Models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RESULTS2_DIRS = ["2048_sample_results2", "../2048_sample_results2", "../../2048_sample_results2"]
DRIVE_DIR  = "/content/drive/MyDrive/KVQuant_v3_Results"
LOCAL_DIRS = [DRIVE_DIR] + RESULTS2_DIRS + ["Data", "../Data", "../../Data"]
REPO_RAW   = "https://raw.githubusercontent.com/yoshikodes/KVCacheCompression/main/2048_sample_results2"

def _local_dir():
    for d in LOCAL_DIRS:
        if os.path.isdir(d) and any(f.endswith("_per_prompt.csv") for f in os.listdir(d)):
            return d
    return None

_LOCAL = _local_dir()

def _read_csv_file(fn):
    """Read one CSV by exact filename from _LOCAL or REPO_RAW. Raises
    FileNotFoundError (local) or requests.HTTPError (remote, typically 404)
    if it doesn't exist there -- callers use that to fall back."""
    if _LOCAL:
        return pd.read_csv(os.path.join(_LOCAL, fn))
    r = requests.get(f"{REPO_RAW}/{fn}", timeout=60)
    r.raise_for_status()
    return pd.read_csv(io.StringIO(r.text))

def _read(cfg, ds):
    """Read one per-prompt CSV; standardize the index column name to 'idx'.
    Also standardize the prompt column to 'prompt' -- some per-prompt CSVs
    (e.g. ones carrying a RULER-style prefilled-prompt column) store it as
    'full_prompt' instead. Some KVQuant runs saved HellaSwag (and RULER) as
    two checkpoint-safety batches instead of one combined file -- if the
    plain filename isn't found, fall back to batch1+batch2 and concatenate.
    IMPORTANT: each batch's raw index column is LOCAL to that batch (both
    restart at 0!), so naively concatenating gives every idx value two rows
    -- silently turning every downstream merge in build_wide() into a
    many-to-many join that MULTIPLIES the row count instead of adding to it
    (e.g. HellaSwag would balloon from 1024 rows to 16384). Offset batch2's
    raw index by batch1's row count so every value is globally unique before
    concatenating -- this matches how the batches were actually sliced
    (items[:N] then items[N:2N])."""
    fn = f"{FNAME[cfg]}_{ds}_per_prompt.csv"
    try:
        df = _read_csv_file(fn)
    except (FileNotFoundError, requests.exceptions.HTTPError):
        parts = []
        offset = 0
        for suffix in ("_batch1", "_batch2"):
            part = _read_csv_file(f"{FNAME[cfg]}_{ds}{suffix}_per_prompt.csv")
            idxc = "question_index" if "question_index" in part.columns else "item_index"
            part[idxc] = part[idxc] + offset
            offset = part[idxc].max() + 1
            parts.append(part)
        df = pd.concat(parts, ignore_index=True)
    idxc = "question_index" if "question_index" in df.columns else "item_index"
    df = df.rename(columns={idxc: "idx"})
    if "prompt" not in df.columns and "full_prompt" in df.columns:
        df = df.rename(columns={"full_prompt": "prompt"})
    return df

print("Data source:", _LOCAL if _LOCAL else REPO_RAW)

# ===========================================================================
# BUILD THE WIDE TABLE: one row per (dataset, prompt); 10 target columns.
# The stored "prompt" column IS the exact text that was fed to the model --
# already fully prefilled (fewshot prefix, answer choices, everything) -- so
# no reconstruction is needed; it's taken as-is from a single reference
# config. The bare question is identical across all 10 configs for a given
# index (the eval order is seed-42 stable across methods), which is what
# makes using a single reference config safe.
# ===========================================================================
def build_wide():
    frames = []
    for ds in DATASETS:
        ref = _read("kvquant_2bit", ds)[["idx", "prompt"]]
        base = pd.DataFrame({"dataset": ds, "idx": ref["idx"].values, "model_input": list(ref["prompt"])})
        for cfg in CONFIGS:                          # attach each config's target column
            d = _read(cfg, ds)[["idx", TARGET]].rename(columns={TARGET: cfg})
            base = base.merge(d, on="idx", how="inner")
        frames.append(base)
    wide = pd.concat(frames, ignore_index=True)
    assert int(wide[CONFIGS].isna().sum().sum()) == 0, "unexpected NaN targets"
    return wide

# ===========================================================================
# FEATURES: cheap numeric descriptions of the prompt text, plus a
# one-hot of the dataset. NOTE: no feature encodes the compression config —
# the config is the OUTPUT axis, so a single prompt maps to one feature row and
# ten predicted values.
# ===========================================================================
def _feats(p):
    s = str(p); t = s.split(); nt = max(len(t), 1); nc = max(len(s), 1)
    return {
        "n_char":      len(s),
        "n_tok":       len(t),
        "ttr":         len(set(t)) / nt,                                   # type-token ratio
        "digit_ratio": sum(c.isdigit() for c in s) / nc,
        "punct_ratio": sum(not c.isalnum() and not c.isspace() for c in s) / nc,
        "upper_ratio": sum(c.isupper() for c in s) / nc,
        "avg_tok":     nc / nt,                                            # mean word length
        "num_count":   len(re.findall(r"\d+", s)),
        "has_q":       int("?" in s),
        "n_newline":   s.count("\n"),
    }

DS_DUMMY_COLS = [f"ds_{d}" for d in DATASETS]        # fixed column order for the one-hot

def fmat(df):
    X = pd.DataFrame([_feats(p) for p in df["model_input"]])
    dummies = pd.get_dummies(df["dataset"], prefix="ds").reindex(columns=DS_DUMMY_COLS, fill_value=0)
    return pd.concat([X.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

# Target matrix (n_prompts, 10), column order == CONFIGS.
def Ymat(df):
    return df[CONFIGS].to_numpy(dtype=float)


Data source: https://raw.githubusercontent.com/yoshikodes/KVCacheCompression/main/Data


## Load, patch prompts, and hold out the test set once

In [ ]:
# ---------------------------------------------------------------------------
# DIAGNOSTIC: confirm every CSV has a unique idx per row before merging.
# build_wide() does an inner merge on "idx" for each of the 10 configs; if
# idx is ever duplicated within one CSV (e.g. a batch1/batch2 concatenation
# gone wrong), the merge silently becomes many-to-many and the row count
# multiplies instead of staying at one row per prompt.
# ---------------------------------------------------------------------------
for ds in DATASETS:
    print(f"\n=== {ds} ===")
    for cfg in CONFIGS:
        d = _read(cfg, ds)
        print(
            cfg,
            "| rows:", len(d),
            "| unique idx:", d["idx"].nunique(),
            "| duplicated rows:", d["idx"].duplicated().sum()
        )


In [3]:
from sklearn.model_selection import train_test_split

# Build the wide table (attaches the 10 targets).
wide = build_wide()
print("wide table:", wide.shape, "| rows per dataset:", wide["dataset"].value_counts().to_dict())

# --------------------------------------------------------------------------
# STEP 1 — hold out a TEST set ONCE, stratified by dataset so the 15% test has
# the same gsm8k/arc/hellaswag mix as the whole. The test set is NOT looked at
# during tuning; it is touched exactly once, at the very end.
# --------------------------------------------------------------------------
tr_parts, te_parts = [], []
for ds, g in wide.groupby("dataset"):
    a, b = train_test_split(g, test_size=TEST_FRAC, random_state=RS, shuffle=True)
    tr_parts.append(a); te_parts.append(b)

trainval_df = pd.concat(tr_parts).sample(frac=1, random_state=RS).reset_index(drop=True)
test_df     = pd.concat(te_parts).sample(frac=1, random_state=RS).reset_index(drop=True)

print("train+val:", len(trainval_df), "| test (held out):", len(test_df))
print("per-dataset train+val:", trainval_df["dataset"].value_counts().to_dict())

# Peek at one patched prompt per dataset so the patching is visible.
for ds in DATASETS:
    ex = wide[wide.dataset == ds]["model_input"].iloc[0]
    print(f"\n--- model_input | {ds} (first {180} chars) ---\n{ex[:180]!r}")


wide table: (3072, 12) | rows per dataset: {'gsm8k': 1024, 'arc_challenge': 1024, 'hellaswag': 1024}
train+val: 2610 | test (held out): 462
per-dataset train+val: {'gsm8k': 870, 'hellaswag': 870, 'arc_challenge': 870}

--- patched model_input | gsm8k (first 180 chars) ---
'You are solving grade-school math word problems.\nShow the calculation step by step, then end with exactly this format:\n#### <final number>\n\nQuestion: There are 15 trees in the grov'

--- patched model_input | arc_challenge (first 180 chars) ---
'An astronomer observes that a planet rotates faster after a meteorite impact. Which is the most likely effect of this increase in rotation?\n\nAnswer choices:\nA. Planetary density wi'

--- patched model_input | hellaswag (first 180 chars) ---
'Playing harmonica: A man is standing in front of a camera. He starts playing a harmonica for the camera. He\n\nAnswer choices:\nA. begins to play the harmonica with his body while loo'


## Targets

In [4]:
# --------------------------------------------------------------------------
# Target is binary `correct` per config. The model outputs a PROBABILITY of
# being correct for each of the 10 configs (predict_proba), so the length-10
# output vector is a vector of probabilities in [0, 1].
# --------------------------------------------------------------------------
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, brier_score_loss

_Y = Ymat(trainval_df)
print("target:", TARGET, "(class balance = P(correct) per config)")
for j, c in enumerate(CONFIGS):
    print(f"  {c:14s} P(correct)={_Y[:,j].mean():.3f}")


target: correct (class balance = P(correct) per config)
  kvquant_2bit   P(correct)=0.548
  kvquant_3bit   P(correct)=0.633
  kvquant_4bit   P(correct)=0.647
  h2o_20         P(correct)=0.553
  h2o_40         P(correct)=0.622
  h2o_60         P(correct)=0.638
  rocketkv_8x    P(correct)=0.456
  rocketkv_16x   P(correct)=0.375
  rocketkv_32x   P(correct)=0.321


## Hyperparameter search — repeated stratified k-fold CV (train+val only)

In [5]:
import lightgbm as lgb
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import RepeatedStratifiedKFold

# ==========================================================================
# STEP 2 -- choose hyperparameters with REPEATED stratified k-fold CV on
# train+val ONLY (folds stratified by dataset), INDEPENDENTLY per config.
# Each of the 10 configs runs its own CV sweep and picks its own best
# hyperparameters -- they are NOT forced to share one setting. Selection
# metric is mean LOG LOSS -- a proper scoring rule for probabilities, so we
# optimize calibrated probability quality, not just 0/1 accuracy. The grid
# is also wider than before, and learning_rate is now searched instead of
# fixed. The test set is NOT touched here.
# ==========================================================================
NUM_LEAVES_GRID    = [7, 10, 15, 20]
MIN_CHILD_GRID     = [50, 100, 200, 300]
N_ESTIMATORS_GRID  = [100, 200, 300, 400]
LEARNING_RATE_GRID = [0.03, 0.1]
CANDIDATES = [{"num_leaves": nl, "min_child_samples": mc, "n_estimators": ne, "learning_rate": lr}
              for nl, mc, ne, lr in itertools.product(
                  NUM_LEAVES_GRID, MIN_CHILD_GRID, N_ESTIMATORS_GRID, LEARNING_RATE_GRID)]

FIXED = dict(subsample=0.8, colsample_bytree=0.8, random_state=RS, verbose=-1, n_jobs=-1)

N_FOLDS, N_REPEATS = 5, 3
print(f"{len(CANDIDATES)} candidates x {N_FOLDS}-fold x {N_REPEATS} repeats "
      f"x {len(CONFIGS)} outputs = {len(CANDIDATES)*N_FOLDS*N_REPEATS*len(CONFIGS)} model fits\n")

class SafeLGBMClassifier(BaseEstimator, ClassifierMixin):
    """Thin LGBMClassifier wrapper with a constant-probability fallback for
    degenerate (single-class) CV folds -- some folds have zero examples of one
    class for a given config, where a real classifier can't be fit. Every
    LightGBM hyperparameter is an explicit __init__ argument (not **kwargs),
    which sklearn.clone()/get_params() require to work correctly. Always
    returns a proper (n_samples, 2) predict_proba, whether or not a real
    classifier was fit, so it drops in wherever an ordinary classifier
    would."""
    def __init__(self, num_leaves=31, min_child_samples=20, n_estimators=100,
                 learning_rate=0.1, subsample=1.0, colsample_bytree=1.0,
                 random_state=None, verbose=-1, n_jobs=-1):
        self.num_leaves = num_leaves
        self.min_child_samples = min_child_samples
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree
        self.random_state = random_state
        self.verbose = verbose
        self.n_jobs = n_jobs

    def _lgbm_params(self):
        return dict(num_leaves=self.num_leaves, min_child_samples=self.min_child_samples,
                    n_estimators=self.n_estimators, learning_rate=self.learning_rate,
                    subsample=self.subsample, colsample_bytree=self.colsample_bytree,
                    random_state=self.random_state, verbose=self.verbose, n_jobs=self.n_jobs)

    def fit(self, X, y):
        y = np.asarray(y).astype(int)
        self.classes_ = np.array([0, 1])
        if len(np.unique(y)) < 2:
            self._const_proba = float(y.mean())          # degenerate fold fallback
            self._model = None
        else:
            self._const_proba = None
            self._model = lgb.LGBMClassifier(**self._lgbm_params()).fit(X, y)
        return self

    def predict_proba(self, X):
        n = len(X)
        if self._model is None:
            p1 = np.full(n, self._const_proba)
        else:
            p1 = self._model.predict_proba(X)[:, 1]
        return np.column_stack([1 - p1, p1])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


class PerTargetSafeLGBMClassifier(BaseEstimator, ClassifierMixin):
    """ONE fitted object wrapping N independently-tuned SafeLGBMClassifier
    models -- one per config -- each using hyperparameters chosen by that
    config's OWN cross-validation (the sweep below) instead of every config
    sharing a single hyperparameter setting. Still ONE fitted object with
    one .fit()/.predict_proba() call, not a hand-managed Python list -- this
    keeps MultiOutputClassifier's external contract without its "one shared
    config for every target" limitation."""
    def __init__(self, per_target_params):
        self.per_target_params = per_target_params   # list of dicts, len == len(CONFIGS)

    def fit(self, X, Y):
        Y = np.asarray(Y).astype(int)
        self.estimators_ = [SafeLGBMClassifier(**params).fit(X, Y[:, j])
                             for j, params in enumerate(self.per_target_params)]
        return self

    def predict_proba(self, X):
        return [est.predict_proba(X) for est in self.estimators_]   # list of (n,2) arrays

    def predict(self, X):
        return np.column_stack([est.predict(X) for est in self.estimators_])


def fit_9(per_target_params, Xtr, Ytr):
    model = PerTargetSafeLGBMClassifier(per_target_params)
    model.fit(Xtr, Ytr.astype(int))
    return model

def proba_9(model, X):
    probas = model.predict_proba(X)          # list of len(CONFIGS) arrays, each (n_samples, 2)
    return np.column_stack([p[:, 1] for p in probas])

def cv_score_per_target(cfg):
    """Returns (n_fold_repeats, len(CONFIGS)) arrays of per-config log loss
    and AUC for this ONE candidate -- NOT averaged across configs, so each
    config's own best hyperparameters can be picked independently below."""
    rskf = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_REPEATS, random_state=RS)
    strat = trainval_df["dataset"].values
    ll_rows, au_rows = [], []
    for tr_idx, va_idx in rskf.split(trainval_df, strat):
        tr, va = trainval_df.iloc[tr_idx], trainval_df.iloc[va_idx]
        Xtr, Xva = fmat(tr).values, fmat(va).values
        Ytr = Ymat(tr).astype(int)
        proba = np.clip(np.column_stack([
            SafeLGBMClassifier(**cfg, **FIXED).fit(Xtr, Ytr[:, j]).predict_proba(Xva)[:, 1]
            for j in range(len(CONFIGS))
        ]), 1e-6, 1 - 1e-6)
        yt = Ymat(va).astype(int)
        ll_row, au_row = [], []
        for j in range(len(CONFIGS)):
            ll_row.append(log_loss(yt[:, j], proba[:, j], labels=[0, 1]))
            au_row.append(roc_auc_score(yt[:, j], proba[:, j]) if len(np.unique(yt[:, j])) == 2 else np.nan)
        ll_rows.append(ll_row); au_rows.append(au_row)
    return np.array(ll_rows), np.array(au_rows)

mean_ll = np.zeros((len(CANDIDATES), len(CONFIGS)))
mean_au = np.zeros((len(CANDIDATES), len(CONFIGS)))
for i, cfg in enumerate(CANDIDATES):
    ll_mat, au_mat = cv_score_per_target(cfg)
    mean_ll[i] = ll_mat.mean(axis=0)
    mean_au[i] = np.nanmean(au_mat, axis=0)
    worst, best = CONFIGS[mean_ll[i].argmax()], CONFIGS[mean_ll[i].argmin()]
    print(f"[{i+1:>3}/{len(CANDIDATES)}] {str(cfg):<70} "
          f"logloss mean={mean_ll[i].mean():.4f}  (worst {worst}={mean_ll[i].max():.4f}, best {best}={mean_ll[i].min():.4f})")

# Select each config's OWN best hyperparameters independently, by lowest
# mean CV log loss (chosen on train+val only).
best_idx_per_config = mean_ll.argmin(axis=0)
best_cfg_per_config = {c: {**CANDIDATES[best_idx_per_config[j]], **FIXED} for j, c in enumerate(CONFIGS)}

print("\nBest hyperparameters per config (selected independently, by mean CV log loss):")
for j, c in enumerate(CONFIGS):
    idx = best_idx_per_config[j]
    print(f"  {c:14s} {CANDIDATES[idx]}  CV logloss={mean_ll[idx, j]:.4f}  CV AUC={mean_au[idx, j]:.4f}")


27 candidates x 5-fold x 3 repeats x 9 outputs = 3645 model fits

[ 1/27] {'num_leaves': 7, 'min_child_samples': 100, 'n_estimators': 100} CV logloss=0.6318±0.0080  AUC=0.6407
[ 2/27] {'num_leaves': 7, 'min_child_samples': 100, 'n_estimators': 200} CV logloss=0.6357±0.0092  AUC=0.6363
[ 3/27] {'num_leaves': 7, 'min_child_samples': 100, 'n_estimators': 300} CV logloss=0.6396±0.0098  AUC=0.6326
[ 4/27] {'num_leaves': 7, 'min_child_samples': 200, 'n_estimators': 100} CV logloss=0.6313±0.0082  AUC=0.6421
[ 5/27] {'num_leaves': 7, 'min_child_samples': 200, 'n_estimators': 200} CV logloss=0.6345±0.0095  AUC=0.6385
[ 6/27] {'num_leaves': 7, 'min_child_samples': 200, 'n_estimators': 300} CV logloss=0.6373±0.0101  AUC=0.6356
[ 7/27] {'num_leaves': 7, 'min_child_samples': 300, 'n_estimators': 100} CV logloss=0.6314±0.0071  AUC=0.6410
[ 8/27] {'num_leaves': 7, 'min_child_samples': 300, 'n_estimators': 200} CV logloss=0.6327±0.0083  AUC=0.6398
[ 9/27] {'num_leaves': 7, 'min_child_samples': 300, 'n

## Final evaluation — test set touched once

In [6]:
# ==========================================================================
# STEP 3 — final fit on ALL train+val with the CV-chosen config; evaluate the
# held-out TEST set exactly once. Save the 10 classifiers + metadata.
# ==========================================================================
import joblib

Xtv, Xte = fmat(trainval_df).values, fmat(test_df).values
Ytv, Yte = Ymat(trainval_df).astype(int), Ymat(test_df).astype(int)

per_target_params = [best_cfg_per_config[c] for c in CONFIGS]
final_model = fit_9(per_target_params, Xtv, Ytv)
proba_tv = np.clip(proba_9(final_model, Xtv), 1e-6, 1 - 1e-6)
proba_te = np.clip(proba_9(final_model, Xte), 1e-6, 1 - 1e-6)

payload = {"task": "classification", "target": TARGET, "configs": CONFIGS,
           "feature_cols": list(fmat(trainval_df).columns),
           "model": final_model, "config": best_cfg_per_config, "seed": RS}
joblib.dump(payload, MODEL_DIR / ARTIFACT)
print("saved", MODEL_DIR / ARTIFACT)
print("hyperparameters per config:")
for c in CONFIGS:
    print(f"  {c:14s} {best_cfg_per_config[c]}")

def per_config_table(Yt, Pt, Yv, Pv):
    rows = []
    for j, c in enumerate(CONFIGS):
        auc_te = roc_auc_score(Yt[:, j], Pt[:, j]) if len(np.unique(Yt[:, j])) == 2 else float("nan")
        auc_tv = roc_auc_score(Yv[:, j], Pv[:, j]) if len(np.unique(Yv[:, j])) == 2 else float("nan")
        rows.append({"config": c,
                     "logloss_test": log_loss(Yt[:, j], Pt[:, j], labels=[0, 1]),
                     "auc_test":     auc_te,
                     "acc_test":     accuracy_score(Yt[:, j], (Pt[:, j] >= 0.5).astype(int)),
                     "brier_test":   brier_score_loss(Yt[:, j], Pt[:, j]),
                     "auc_trainval": auc_tv})
    tab = pd.DataFrame(rows)
    tab.loc[len(tab)] = {"config": "MEAN",
        "logloss_test": tab.logloss_test.mean(), "auc_test": tab.auc_test.mean(),
        "acc_test": tab.acc_test.mean(), "brier_test": tab.brier_test.mean(),
        "auc_trainval": tab.auc_trainval.mean()}
    return tab

tab = per_config_table(Yte, proba_te, Ytv, proba_tv)
print("\n=== PER-CONFIG RESULTS (mean over the 10-vector at the bottom) ===")
print(tab.round(4).to_string(index=False))
print(f"\ngeneralization gap (trainval AUC - test AUC): "
      f"{tab.iloc[-1].auc_trainval - tab.iloc[-1].auc_test:.4f}  (small = good)")


saved artifacts/clf_correct_lightgbm.joblib | config: {'num_leaves': 7, 'min_child_samples': 200, 'n_estimators': 100}

=== PER-CONFIG RESULTS (mean over the 9-vector at the bottom) ===
      config  logloss_test  auc_test  acc_test  brier_test  auc_trainval
kvquant_2bit        0.5869    0.7531    0.6688      0.2008        0.7332
kvquant_3bit        0.6144    0.7051    0.6126      0.2144        0.7000
kvquant_4bit        0.6024    0.6964    0.5931      0.2098        0.7056
      h2o_20        0.6187    0.7117    0.6732      0.2148        0.7322
      h2o_40        0.6080    0.7034    0.6407      0.2114        0.7017
      h2o_60        0.6006    0.7139    0.6364      0.2085        0.6911
 rocketkv_8x        0.6705    0.5931    0.6082      0.2388        0.6737
rocketkv_16x        0.6098    0.6098    0.6883      0.2100        0.6908
rocketkv_32x        0.5885    0.6278    0.7056      0.2002        0.6551
        MEAN        0.6111    0.6794    0.6474      0.2121        0.6982

generaliza

## Inference — the length-10 output vector

In [7]:
# ==========================================================================
# INFERENCE HELPER — one prompt in, a length-10 probability vector out.
# Each entry is P(answered correctly) under that config; order == CONFIGS.
# ==========================================================================
def predict_vector(prompt_text, dataset):
    """dataset in {'gsm8k','arc_challenge','hellaswag'} — used only for the one-hot."""
    row = pd.DataFrame({"model_input": [str(prompt_text)], "dataset": [dataset]})
    X = fmat(row).values
    vec = proba_9(final_model, X)[0]
    return dict(zip(CONFIGS, vec))

_ex = test_df.iloc[0]
print("dataset:", _ex["dataset"], "| P(correct) per config")
pred_vec = predict_vector(_ex["model_input"], _ex["dataset"])
for c in CONFIGS:
    print(f"  {c:14s} P(correct)={pred_vec[c]:.3f}   actual_correct={int(_ex[c])}")


dataset: gsm8k | P(correct) per config
  kvquant_2bit   P(correct)=0.237   actual_correct=1
  kvquant_3bit   P(correct)=0.409   actual_correct=0
  kvquant_4bit   P(correct)=0.462   actual_correct=1
  h2o_20         P(correct)=0.393   actual_correct=1
  h2o_40         P(correct)=0.441   actual_correct=1
  h2o_60         P(correct)=0.443   actual_correct=1
  rocketkv_8x    P(correct)=0.451   actual_correct=1
  rocketkv_16x   P(correct)=0.344   actual_correct=1
  rocketkv_32x   P(correct)=0.221   actual_correct=0


## How to read this

- The **CV table** is where hyperparameters are chosen — by mean validation score across all folds *and* repeats, with the std as an error bar. If two configs are within ~1 std, prefer the simpler (more regularized) one.
- The **per-config table** reports the held-out test result for each of the 10 configurations, with a **MEAN** row summarizing the length-10 output vector. The test set influenced neither the features nor the hyperparameters, so it is an honest estimate.
- `predict_vector(prompt, dataset)` shows the end use: one prompt in, the length-10 vector out.
- The saved `/content/drive/MyDrive/KV_Cache_Models/*.joblib` bundles the model (one `PerTargetSafeLGBMClassifier` object emitting all 10 probabilities, each with its own independently-tuned hyperparameters), the feature-column order, the chosen config-per-target dict and the target space, so predictions can be reproduced elsewhere.